# Rearmament on NATO's Eastern Flank (v2 — Referee-Hardened Empirical Pipeline)
**Target Journal:** *Eastern European Economics* (Taylor & Francis, SSCI)  
**Author:** Murat Şahin — Department of Public Finance, Faculty of Economics, Istanbul University  
**v2 Upgrades:**  
1. SDID **Block Bootstrap SE** (Arkhangelsky et al., 2021 Algorithm 2, B=400) + **Jackknife SE** (Algorithm 3) + **95% CIs** alongside Placebo SE.  
2. Explicit **SDID Unit Weights ($\hat{\omega}_i$) & Time Weights ($\hat{\lambda}_t$)** diagnostics for transparency.  
3. **Denominator-Free Budget-Share Local Projections** ($\Delta$ Defense Share of Total Expenditure $	o$ Civilian Budget Shares) with **Wild Cluster Bootstrap** ($B=1,000$) to eliminate common-GDP-denominator recession bias.  
4. Publication-grade 300-DPI figures with **clean integer year ticks** (zero decimal years).


In [ ]:
# ==============================================================================
# CELL 1: IMPORTS, COUNTRY GROUPS & LIVE API FETCHERS (EUROSTAT + IMF + WDI)
# ==============================================================================
import warnings, time, requests
import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import minimize
import statsmodels.api as sm
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

warnings.filterwarnings("ignore")
np.random.seed(42)

ISO2_TO_ISO3 = {
    "BG": "BGR", "HR": "HRV", "CZ": "CZE", "EE": "EST", "HU": "HUN",
    "LV": "LVA", "LT": "LTU", "PL": "POL", "RO": "ROU", "SK": "SVK", "SI": "SVN",
    "AT": "AUT", "BE": "BEL", "CY": "CYP", "DK": "DNK", "FI": "FIN",
    "FR": "FRA", "DE": "DEU", "EL": "GRC", "IE": "IRL", "IT": "ITA",
    "LU": "LUX", "MT": "MLT", "NL": "NLD", "PT": "PRT", "ES": "ESP", "SE": "SWE"
}

CEE11 = ["BG", "HR", "CZ", "EE", "HU", "LV", "LT", "PL", "RO", "SK", "SI"]
FRONTLINE = ["PL", "EE", "LV", "LT", "RO", "SK", "HU"]
CEE_EURO = ["EE", "LV", "LT", "SK", "SI", "HR"]
CEE_NONEURO = ["PL", "CZ", "HU", "RO", "BG"]
WEST16 = [c for c in ISO2_TO_ISO3 if c not in CEE11]
ALL_EU27 = CEE11 + WEST16
YEARS_MIN, YEARS_MAX, TREAT_YEAR = 2005, 2024, 2022

def fetch_eurostat_jsonstat(dataset, params_dict, retries=3):
    base = f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/{dataset}"
    params = [("format", "JSON"), ("lang", "en")]
    for k, v in params_dict.items():
        if isinstance(v, list):
            for item in v: params.append((k, item))
        else:
            params.append((k, v))
    for attempt in range(retries):
        try:
            r = requests.get(base, params=params, timeout=60)
            r.raise_for_status()
            data = r.json()
            dims, sizes = data["id"], data["size"]
            idx_to_key = {d: {v: k for k, v in data["dimension"][d]["category"]["index"].items()} for d in dims}
            strides = [1] * len(dims)
            for i in range(len(dims) - 2, -1, -1):
                strides[i] = strides[i + 1] * sizes[i + 1]
            rows = []
            for flat_str, val in data.get("value", {}).items():
                if val is None: continue
                flat, rec = int(flat_str), {"value": float(val)}
                rem = flat
                for i, d in enumerate(dims):
                    rec[d] = idx_to_key[d][rem // strides[i]]
                    rem = rem % strides[i]
                rows.append(rec)
            return pd.DataFrame(rows)
        except Exception as e:
            if attempt == retries - 1: raise RuntimeError(f"Eurostat failed for {dataset}: {e}")
            time.sleep(2)

print("Fetching Eurostat COFOG (gov_10a_exp)...")
df_cofog = fetch_eurostat_jsonstat("gov_10a_exp", {
    "unit": "PC_GDP", "sector": "S13", "na_item": ["TE", "P51G"],
    "cofog99": ["TOTAL", "GF02", "GF04", "GF07", "GF09", "GF10"], "geo": ALL_EU27
})
df_cofog["year"] = df_cofog["time"].astype(int)
df_cofog = df_cofog[(df_cofog["year"] >= YEARS_MIN) & (df_cofog["year"] <= YEARS_MAX)].copy()

te_piv = df_cofog[df_cofog["na_item"] == "TE"].pivot_table(index=["geo", "year"], columns="cofog99", values="value", aggfunc="first").reset_index()
te_piv = te_piv.rename(columns={"TOTAL": "tot_exp", "GF02": "defense", "GF04": "econ_affairs", "GF07": "health", "GF09": "education", "GF10": "social_prot"})
inv_piv = df_cofog[df_cofog["na_item"] == "P51G"].pivot_table(index=["geo", "year"], columns="cofog99", values="value", aggfunc="first").reset_index()
inv_piv = inv_piv.rename(columns={"TOTAL": "pub_inv", "GF02": "def_inv"})

print("Fetching Eurostat gov_10a_main...")
df_main = fetch_eurostat_jsonstat("gov_10a_main", {"unit": "PC_GDP", "sector": "S13", "na_item": ["TR", "B9", "D41PAY"], "geo": ALL_EU27})
df_main["year"] = df_main["time"].astype(int)
df_main = df_main[(df_main["year"] >= YEARS_MIN) & (df_main["year"] <= YEARS_MAX)].copy()
main_piv = df_main.pivot_table(index=["geo", "year"], columns="na_item", values="value", aggfunc="first").reset_index()
main_piv = main_piv.rename(columns={"TR": "tot_rev", "B9": "nlb_gdp", "D41PAY": "int_pay"})

panel = te_piv.merge(inv_piv[["geo", "year"] + [c for c in ["pub_inv", "def_inv"] if c in inv_piv.columns]], on=["geo", "year"], how="left")
panel = panel.merge(main_piv, on=["geo", "year"], how="left")
panel["iso3"] = panel["geo"].map(ISO2_TO_ISO3)

def fetch_imf_indicator(code):
    r = requests.get(f"https://www.imf.org/external/datamapper/api/v1/{code}", timeout=45)
    r.raise_for_status()
    vals = r.json().get("values", {}).get(code, {})
    iso3_set = set(ISO2_TO_ISO3.values())
    rows = [{"iso3": k, "year": int(y), code: float(v)} for k, yd in vals.items() if k in iso3_set for y, v in yd.items() if YEARS_MIN <= int(y) <= YEARS_MAX and v is not None]
    return pd.DataFrame(rows)

print("Fetching IMF WEO DataMapper...")
for imf_code, col_name in [("GGXWDG_NGDP", "debt_gdp"), ("NGDP_RPCH", "gdp_growth"), ("PCPIPCH", "cpi_inf")]:
    panel = panel.merge(fetch_imf_indicator(imf_code).rename(columns={imf_code: col_name}), on=["iso3", "year"], how="left")

def fetch_wdi(indicator, col_name):
    url = f"https://api.worldbank.org/v2/country/{';'.join(ISO2_TO_ISO3.values())}/indicator/{indicator}?format=json&per_page=5000&date={YEARS_MIN}:{YEARS_MAX}"
    data = requests.get(url, timeout=45).json()
    rows = [{"iso3": it["countryiso3code"], "year": int(it["date"]), col_name: float(it["value"])} for it in (data[1] if len(data) > 1 and data[1] else []) if it["value"] is not None]
    return pd.DataFrame(rows)

print("Fetching World Bank WDI (SIPRI & Unemployment)...")
panel = panel.merge(fetch_wdi("MS.MIL.XPND.GD.ZS", "sipri_mil"), on=["iso3", "year"], how="left")
panel = panel.merge(fetch_wdi("SL.UEM.TOTL.ZS", "unemp_rate"), on=["iso3", "year"], how="left")

# Derived Level & Budget-Share Variables (Budget Shares = Denominator-Free!)
panel["social_core"] = panel["health"] + panel["education"] + panel["social_prot"]
panel["non_def_inv"] = panel["pub_inv"] - panel["def_inv"].fillna(0.0) if "def_inv" in panel.columns else panel["pub_inv"]
panel["primary_bal"] = panel["nlb_gdp"] + panel["int_pay"].fillna(0.0) if "int_pay" in panel.columns else panel["nlb_gdp"]
for col, scol in [("defense", "sh_defense"), ("social_core", "sh_social_core"), ("social_prot", "sh_social_prot"),
                  ("health", "sh_health"), ("education", "sh_education"), ("econ_affairs", "sh_econ_affairs"), ("pub_inv", "sh_pub_inv")]:
    panel[scol] = 100.0 * panel[col] / panel["tot_exp"]

panel["treated"] = panel["geo"].isin(CEE11).astype(int)
panel["post"] = (panel["year"] >= TREAT_YEAR).astype(int)
panel = panel.sort_values(["geo", "year"]).reset_index(drop=True)
yr_counts = panel.groupby("year")["defense"].count()
MAX_COMPLETE_YEAR = int(yr_counts[yr_counts >= 25].index.max())
panel = panel[panel["year"] <= MAX_COMPLETE_YEAR].copy()
print(f"FINAL PANEL READY: N={panel['geo'].nunique()}, Years={panel['year'].min()}-{MAX_COMPLETE_YEAR}, Obs={len(panel)}")


In [ ]:
# ==============================================================================
# CELL 2: SDID WITH BLOCK BOOTSTRAP SE (ALG. 2), JACKKNIFE SE (ALG. 3),
#         PLACEBO SE (ALG. 4), 95% CIs, AND WEIGHT DIAGNOSTICS
# ==============================================================================
summary_lines = []
def log(msg=""):
    print(msg)
    summary_lines.append(str(msg))

log("=" * 88)
log("STAGE 1 v2 REFEREE-HARDENED RESULTS — EASTERN EUROPEAN ECONOMICS (NATO REARMAMENT)")
log(f"Sample: EU-27 (N=11 CEE Treated, N=16 West Control), Years 2005-{MAX_COMPLETE_YEAR}")
log("=" * 88)

def compute_sdid_weights(Y_ctrl_pre, Y_tr_pre, Y_ctrl_post):
    N0, T0 = Y_ctrl_pre.shape
    N1, T1 = Y_tr_pre.shape[0], Y_ctrl_post.shape[1]
    sigma_hat = np.std(np.diff(Y_ctrl_pre, axis=1), ddof=1)
    zeta = ((N1 * T1) ** 0.25) * sigma_hat
    zeta_lam = 1e-6 * sigma_hat

    y_tr_mean_pre = Y_tr_pre.mean(axis=0)
    def obj_w(p):
        return np.sum((p[0] + p[1:] @ Y_ctrl_pre - y_tr_mean_pre) ** 2) + (zeta ** 2) * T0 * np.sum(p[1:] ** 2)
    res_w = minimize(obj_w, np.concatenate([[0.0], np.ones(N0)/N0]), method="SLSQP",
                     bounds=[(None, None)] + [(0.0, 1.0)]*N0, constraints=[{"type": "eq", "fun": lambda p: np.sum(p[1:]) - 1.0}],
                     options={"ftol": 1e-10, "maxiter": 400})
    omega = np.maximum(res_w.x[1:], 0.0); omega /= omega.sum()

    y_co_mean_post = Y_ctrl_post.mean(axis=1)
    def obj_l(p):
        return np.sum((p[0] + Y_ctrl_pre @ p[1:] - y_co_mean_post) ** 2) + (zeta_lam ** 2) * N0 * np.sum(p[1:] ** 2)
    res_l = minimize(obj_l, np.concatenate([[0.0], np.ones(T0)/T0]), method="SLSQP",
                     bounds=[(None, None)] + [(0.0, 1.0)]*T0, constraints=[{"type": "eq", "fun": lambda p: np.sum(p[1:]) - 1.0}],
                     options={"ftol": 1e-10, "maxiter": 400})
    lam = np.maximum(res_l.x[1:], 0.0); lam /= lam.sum()
    return omega, lam

def sdid_att_only(Y_co_pre, Y_co_post, Y_tr_pre, Y_tr_post):
    w, l = compute_sdid_weights(Y_co_pre, Y_tr_pre, Y_co_post)
    return float((Y_tr_post.mean() - (Y_tr_pre @ l).mean()) - ((w @ Y_co_post).mean() - float(w @ Y_co_pre @ l))), w, l

def sdid_full_inference(df_in, outcome, treated_list, control_list, pre_start=2005, treat_year=2022, B_boot=400, B_plac=300):
    sub = df_in[df_in["year"] >= pre_start].pivot(index="geo", columns="year", values=outcome)
    tr_u = [u for u in treated_list if u in sub.index and sub.loc[u].notna().all()]
    co_u = [u for u in control_list if u in sub.index and sub.loc[u].notna().all()]
    pre_y = [y for y in sub.columns if y < treat_year]
    post_y = [y for y in sub.columns if y >= treat_year]
    Y_tr_pre, Y_tr_post = sub.loc[tr_u, pre_y].values, sub.loc[tr_u, post_y].values
    Y_co_pre, Y_co_post = sub.loc[co_u, pre_y].values, sub.loc[co_u, post_y].values

    att, omega, lam = sdid_att_only(Y_co_pre, Y_co_post, Y_tr_pre, Y_tr_post)
    syn_s = omega @ sub.loc[co_u].values
    tr_s  = sub.loc[tr_u].mean(axis=0).values
    pre_rmse = float(np.sqrt(np.mean(((tr_s[:len(pre_y)] - tr_s[:len(pre_y)].mean()) - (syn_s[:len(pre_y)] - syn_s[:len(pre_y)].mean()))**2)))

    # 1. Block Bootstrap SE (Arkhangelsky et al. Algorithm 2: resample treated & control units with replacement)
    rng = np.random.default_rng(42)
    N1, N0 = len(tr_u), len(co_u)
    boot_atts = []
    for _ in range(B_boot):
        it = rng.integers(0, N1, size=N1)
        ic = rng.integers(0, N0, size=N0)
        try:
            a_b, _, _ = sdid_att_only(Y_co_pre[ic], Y_co_post[ic], Y_tr_pre[it], Y_tr_post[it])
            boot_atts.append(a_b)
        except Exception:
            pass
    se_boot = float(np.std(boot_atts, ddof=1))
    p_boot  = float(2 * (1 - stats.norm.cdf(abs(att / se_boot)))) if se_boot > 0 else np.nan
    ci_lo, ci_hi = att - 1.96 * se_boot, att + 1.96 * se_boot

    # 2. Jackknife SE (Arkhangelsky et al. Algorithm 3: leave-one-unit-out holding weights fixed)
    jk_atts = []
    for i in range(N1):
        m = np.arange(N1) != i
        jk_atts.append(float((Y_tr_post[m].mean() - (Y_tr_pre[m] @ lam).mean()) - ((omega @ Y_co_post).mean() - float(omega @ Y_co_pre @ lam))))
    for j in range(N0):
        m = np.arange(N0) != j
        wm = omega[m] / omega[m].sum() if omega[m].sum() > 0 else np.ones(N0-1)/(N0-1)
        jk_atts.append(float((Y_tr_post.mean() - (Y_tr_pre @ lam).mean()) - ((wm @ Y_co_post[m]).mean() - float(wm @ Y_co_pre[m] @ lam))))
    N_tot = N1 + N0
    se_jk = float(np.sqrt(((N_tot - 1) / N_tot) * np.sum((np.array(jk_atts) - np.mean(jk_atts))**2)))
    p_jk  = float(2 * (1 - stats.norm.cdf(abs(att / se_jk)))) if se_jk > 0 else np.nan

    # 3. Placebo SE (Algorithm 4)
    rng_p = np.random.default_rng(123)
    n_pseudo = min(N1, max(1, N0 - 3))
    plac_atts = []
    for _ in range(B_plac):
        idx_t = rng_p.choice(N0, size=n_pseudo, replace=False)
        idx_c = np.setdiff1d(np.arange(N0), idx_t)
        try:
            a_p, _, _ = sdid_att_only(Y_co_pre[idx_c], Y_co_post[idx_c], Y_co_pre[idx_t], Y_co_post[idx_t])
            plac_atts.append(a_p)
        except Exception:
            pass
    se_plac = float(np.std(plac_atts, ddof=1))
    p_plac  = float(2 * (1 - stats.norm.cdf(abs(att / se_plac)))) if se_plac > 0 else np.nan

    return {
        "att": att, "se_boot": se_boot, "p_boot": p_boot, "ci_lo": ci_lo, "ci_hi": ci_hi,
        "se_jk": se_jk, "p_jk": p_jk, "se_plac": se_plac, "p_plac": p_plac, "pre_rmse": pre_rmse,
        "omega": dict(zip(co_u, omega)), "lam": dict(zip(pre_y, lam)),
        "years": list(sub.columns), "tr_series": tr_s, "syn_series": syn_s
    }

OUTCOMES = [
    ("defense",      "Defense (COFOG GF02, % GDP)"),
    ("sipri_mil",    "Defense (SIPRI, % GDP)"),
    ("social_core",  "Core Social (GF07+GF09+GF10)"),
    ("social_prot",  "Social Protection (GF10)"),
    ("health",       "Health (GF07)"),
    ("education",    "Education (GF09)"),
    ("econ_affairs", "Economic Affairs (GF04)"),
    ("pub_inv",      "Public Investment (P51G)"),
    ("non_def_inv",  "Non-Defense Public Inv."),
    ("sh_social_core","Core Social Share (% Tot.Exp.)"),
    ("tot_exp",      "Total Gov. Expenditure"),
    ("tot_rev",      "Total Gov. Revenue"),
    ("nlb_gdp",      "Fiscal Balance (B9)"),
    ("primary_bal",  "Primary Balance"),
    ("debt_gdp",     "Gross Public Debt"),
]

log("\n--- TABLE 2 (v2): SDID WITH BLOCK BOOTSTRAP SE, JACKKNIFE SE, PLACEBO SE & 95% BOOT CI ---")
log(f"{'Outcome':<32} {'ATT':>7} | {'BootSE':>6} {'BootP':>6} {'95% Boot CI':>17} | {'JK SE':>6} {'JK p':>6} | {'PlacSE':>6} {'PlacP':>6}")
sdid_v2 = {}
for col, label in OUTCOMES:
    if col not in panel.columns or panel[col].isna().mean() > 0.25: continue
    r = sdid_full_inference(panel, col, CEE11, WEST16, pre_start=2005, treat_year=TREAT_YEAR, B_boot=400, B_plac=300)
    sdid_v2[col] = r
    ci_s = f"[{r['ci_lo']:+.2f}, {r['ci_hi']:+.2f}]"
    log(f"{label:<32} {r['att']:+7.3f} | {r['se_boot']:6.3f} {r['p_boot']:6.3f} {ci_s:>17} | {r['se_jk']:6.3f} {r['p_jk']:6.3f} | {r['se_plac']:6.3f} {r['p_plac']:6.3f}")

log("\n--- SDID WEIGHT DIAGNOSTICS (TOP-5 UNIT WEIGHTS omega_i & TOP-4 TIME WEIGHTS lambda_t) ---")
for col in ["defense", "social_core", "pub_inv", "nlb_gdp"]:
    r = sdid_v2[col]
    top_w = sorted(r["omega"].items(), key=lambda x: -x[1])[:5]
    top_l = sorted(r["lam"].items(), key=lambda x: -x[1])[:4]
    w_str = ", ".join([f"{k}:{v:.2f}" for k, v in top_w])
    l_str = ", ".join([f"{k}:{v:.2f}" for k, v in top_l])
    log(f"  {col:<12} Top Units: [{w_str}] || Top Pre-Years: [{l_str}]")


In [ ]:
# ==============================================================================
# CELL 3: TABLE 4 (v2) — DENOMINATOR-FREE BUDGET-SHARE LOCAL PROJECTIONS
#         + WILD CLUSTER BOOTSTRAP (B=1000) FOR SMALL CLUSTER COUNT (G=11)
# ==============================================================================
def panel_lp_wild_boot(df_in, y_col, shock_var, horizons=[0, 1, 2], sample_geos=CEE11, B=1000):
    d = df_in[df_in["geo"].isin(sample_geos)].sort_values(["geo", "year"]).copy()
    d["d_shock"] = d.groupby("geo")[shock_var].diff()
    d["l_debt"]  = d.groupby("geo")["debt_gdp"].shift(1)
    out = {}
    for h in horizons:
        d["y_cum"] = d.groupby("geo")[y_col].shift(-h) - d.groupby("geo")[y_col].shift(1)
        d["l_y"]   = d.groupby("geo")[y_col].shift(1)
        sub = d[["geo", "year", "y_cum", "d_shock", "gdp_growth", "unemp_rate", "l_debt", "l_y"]].dropna().copy()
        fe_geo = pd.get_dummies(sub["geo"], drop_first=True, dtype=float)
        fe_yr  = pd.get_dummies(sub["year"], drop_first=True, dtype=float)
        X = pd.concat([pd.Series(1.0, index=sub.index, name="const"),
                       sub[["d_shock", "gdp_growth", "unemp_rate", "l_debt", "l_y"]], fe_geo, fe_yr], axis=1)
        y = sub["y_cum"].values
        mod = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": sub["geo"]})
        b_obs = float(mod.params["d_shock"]); se_obs = float(mod.bse["d_shock"]); t_obs = b_obs / se_obs

        # Restricted Wild Cluster Bootstrap (Rademacher weights, G=11)
        X_r = X.drop(columns=["d_shock"])
        mod_r = sm.OLS(y, X_r).fit()
        yh_r, e_r = mod_r.fittedvalues.values, mod_r.resid.values
        X_mat = X.values
        XtX_inv = np.linalg.pinv(X_mat.T @ X_mat)
        s_idx = list(X.columns).index("d_shock")
        clusters = sub["geo"].unique(); cl_map = sub["geo"].values
        G, N, K = len(clusters), len(y), X_mat.shape[1]
        rng = np.random.default_rng(777 + h)
        t_boots = []
        for _ in range(B):
            w_cl = {g: (1.0 if rng.random() > 0.5 else -1.0) for g in clusters}
            y_b = yh_r + np.array([w_cl[g] for g in cl_map]) * e_r
            bh = XtX_inv @ (X_mat.T @ y_b)
            eb = y_b - X_mat @ bh
            meat = sum(float((X_mat[cl_map==g].T @ eb[cl_map==g]) @ XtX_inv[:, s_idx])**2 for g in clusters)
            seb = np.sqrt((G / (G - 1)) * ((N - 1) / (N - K)) * meat)
            if seb > 0: t_boots.append(bh[s_idx] / seb)
        p_wild = float(np.mean(np.abs(t_boots) >= abs(t_obs)))
        out[h] = (b_obs, se_obs, float(mod.pvalues["d_shock"]), p_wild)
    return out

log("\n--- TABLE 4A (v2): % GDP LOCAL PROJECTIONS WITH WILD CLUSTER BOOTSTRAP (G=11, B=1000) ---")
log(f"{'Outcome (% GDP)':<16} {'h=0 (SE, Wild p)':<27} {'h=1 (SE, Wild p)':<27} {'h=2 (SE, Wild p)':<27}")
for col in ["social_core", "social_prot", "health", "education", "econ_affairs", "pub_inv", "tot_exp", "nlb_gdp"]:
    r = panel_lp_wild_boot(panel, col, "defense")
    log(f"{col:<16} {r[0][0]:+.3f} ({r[0][1]:.3f}, wp={r[0][3]:.3f})   {r[1][0]:+.3f} ({r[1][1]:.3f}, wp={r[1][3]:.3f})   {r[2][0]:+.3f} ({r[2][1]:.3f}, wp={r[2][3]:.3f})")

log("\n--- TABLE 4B (v2): DENOMINATOR-FREE BUDGET-SHARE LOCAL PROJECTIONS (d Defense Share -> Civilian Shares, Wild p) ---")
log(f"{'Outcome (% TE)':<16} {'h=0 (SE, Wild p)':<27} {'h=1 (SE, Wild p)':<27} {'h=2 (SE, Wild p)':<27}")
for col in ["sh_social_core", "sh_social_prot", "sh_health", "sh_education", "sh_econ_affairs", "sh_pub_inv"]:
    r = panel_lp_wild_boot(panel, col, "sh_defense")
    log(f"{col:<16} {r[0][0]:+.3f} ({r[0][1]:.3f}, wp={r[0][3]:.3f})   {r[1][0]:+.3f} ({r[1][1]:.3f}, wp={r[1][3]:.3f})   {r[2][0]:+.3f} ({r[2][1]:.3f}, wp={r[2][3]:.3f})")

# Frontline (N=7) Block Bootstrap SDID Verification
log("\n--- TABLE 5 ADDENDUM: FRONTLINE (N=7) & EURO/NON-EURO SDID BLOCK BOOTSTRAP p-VALUES ---")
for sname, tr_l in [("Frontline (N=7)", FRONTLINE), ("Eurozone CEE (N=6)", CEE_EURO), ("Non-Eurozone CEE (N=5)", CEE_NONEURO)]:
    vals = []
    for col in ["defense", "social_core", "pub_inv", "nlb_gdp"]:
        r = sdid_full_inference(panel, col, tr_l, WEST16, pre_start=2005, treat_year=TREAT_YEAR, B_boot=300, B_plac=200)
        vals.append(f"{r['att']:+.2f}(bp={r['p_boot']:.2f})")
    log(f"  {sname:<24} Def:{vals[0]:<15} Soc:{vals[1]:<15} Inv:{vals[2]:<15} Bal:{vals[3]:<15}")

with open("stage1_eee_nato_summary_v2.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))
print("\nSaved: stage1_eee_nato_summary_v2.txt")


In [ ]:
# ==============================================================================
# CELL 4: CLEAN 300-DPI FIGURE 1 (INTEGER YEAR TICKS — ZERO DECIMAL YEARS!)
# ==============================================================================
plt.rcParams.update({"font.family": "DejaVu Serif", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.8), dpi=300)
fig1_panels = [
    ("defense",     "A. Defense Expenditure (COFOG GF02, % GDP)"),
    ("social_core", "B. Core Social Spending (Health+Educ+SocProt, % GDP)"),
    ("pub_inv",     "C. Public Investment (GFCF P51G, % GDP)"),
    ("nlb_gdp",     "D. General Government Fiscal Balance (% GDP)"),
]
for ax, (col, title) in zip(axes.flat, fig1_panels):
    st = sdid_v2[col]
    yrs = [int(y) for y in st["years"]]
    pre_mask = np.array(yrs) < TREAT_YEAR
    shift = st["tr_series"][pre_mask].mean() - st["syn_series"][pre_mask].mean()
    ax.plot(yrs, st["tr_series"], color="#1B365D", lw=2.2, marker="o", ms=3.8, label="NATO Eastern Flank (CEE-11)")
    ax.plot(yrs, st["syn_series"] + shift, color="#C85A17", lw=2.0, ls="--", marker="s", ms=3.5, label="Synthetic Control (SDID)")
    ax.axvline(2021.5, color="#555555", ls=":", lw=1.4)
    ymin, ymax = ax.get_ylim()
    ax.text(2021.65, ymax - 0.09*(ymax - ymin), "2022 Shock", fontsize=8.5, color="#444444")
    ax.set_title(title, fontsize=10.5, fontweight="bold", loc="left")
    ax.set_xlabel("Year", fontsize=9)
    ax.set_ylabel("% of GDP", fontsize=9)
    ax.set_xticks([2005, 2008, 2011, 2014, 2017, 2020, 2024])
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(axis="y", alpha=0.25)
    ax.text(0.03, 0.06, f"SDID ATT = {st['att']:+.2f} pp (Boot p = {st['p_boot']:.3f})", transform=ax.transAxes,
            fontsize=8.5, bbox=dict(boxstyle="round,pad=0.3", fc="#F7F7F7", ec="#CCCCCC"))

axes[0, 0].legend(frameon=False, fontsize=8.5, loc="upper left")
plt.suptitle("Synthetic Difference-in-Differences: Post-2022 Rearmament and Budgetary Trade-Offs in CEE-11", fontsize=12, fontweight="bold", y=0.99)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("fig1_sdid_rearmament_cee_v2.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: fig1_sdid_rearmament_cee_v2.png")

try:
    from google.colab import files
    for fn in ["stage1_eee_nato_summary_v2.txt", "fig1_sdid_rearmament_cee_v2.png"]:
        files.download(fn)
except ImportError:
    pass
